<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## 2-opt and Local Optimality

**Improve complete tours and identify exactly what a local stopping condition proves.**

Use the same five-city delivery instance. The clock model, calibration, and correction bound do not apply; the decision is now a discrete visit order. Keep costs and requirements fixed; compare segment reversals with a smaller neighborhood.

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 1 · The delivery problem

A vehicle visits cities $0,1,2,3,4$ once each and returns to its start. The fixed symmetric costs $C_{ij}$ are in **distance units**; they are not assumed to be map distances.

| From / to | $0$ | $1$ | $2$ | $3$ | $4$ |
|:---|---:|---:|---:|---:|---:|
| $0$ | 0 | 2 | 9 | 10 | 7 |
| $1$ | 2 | 0 | 8 | 5 | 6 |
| $2$ | 9 | 8 | 0 | 4 | 3 |
| $3$ | 10 | 5 | 4 | 0 | 1 |
| $4$ | 7 | 6 | 3 | 1 | 0 |

The decision is the tour $x=(v_0,\ldots,v_5)$. Feasibility requires that $v_0,\ldots,v_4$ contain every city once and $v_5=v_0$. Its objective is total distance $f(x)=\sum_{k=0}^4C_{v_k,v_{k+1}}$. Reject an infeasible route before comparing scores.

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 2 · One move changes two connections

A **2-opt neighbor** reverses internal positions $i$ through $j$, $1\le i<j\le4$. The endpoints remain fixed. Reversing all four internal positions only reverses the same cycle, so exclude that redundant move; five neighbors remain.

From $x=(1,0,4,3,2,1)$ with distance $22$, reverse $(3,2)$ to obtain $(1,0,4,2,3,1)$ with distance $21$. The removed boundary edges cost $1+8=9$; the added ones cost $3+5=8$. Internal reversed edges retain their costs because the matrix is symmetric.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/05-1_heuristic_optimization_greedy_and_local_search/assets/two_opt_segment_reversal-v3.png" alt="Reversing cities 3 and 2 changes the boundary connections and lowers total distance from 22 to 21." width="512" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 3 · Scan, improve, and verify the stopping condition

Best improvement evaluates every neighbor, accepts the shortest only if it strictly improves the current tour, and repeats. Equal best moves follow the listing order. Here the accepted distances are $22\to21\to20$.

A complete final scan with no improvement establishes

$$
f(x)\le f(x')\quad\text{for every }x'\in N(x),
$$

where $N(x)$ is the chosen neighborhood. This is local optimality for 2-opt. A work limit reached before that scan would not establish it. Strict improvement on finitely many tours ensures termination; accepting equal-cost moves could cycle.

The last column still evaluates every neighbor. It stops because all are worse than the current $20$, not because a previously known target was reached.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/05-1_heuristic_optimization_greedy_and_local_search/assets/local_search_complete_scans.png" alt="Complete neighbor scans accept 21 and then 20; a final scan finds no improving 2-opt neighbor." width="512" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 4 · Local optimality depends on the neighborhood

Fix the tour $(2,3,1,0,4,2)$ with distance $21$. Swaps of consecutive internal positions give distances $33$, $25$, and $31$, so the tour is locally optimal under that restricted rule. Full 2-opt adds a reversal of positions 1 through 3, giving $(2,0,1,3,4,2)$ with distance $20$.

| Method | Evidence supplied |
|:---|:---|
| Greedy construction | A feasible tour; starts $0$–$4$ give $20,22,20,24,22$ |
| Greedy followed by 2-opt | All five runs reach $20$; a final scan verifies local optimality |
| Enumeration of all 24 written tours | Global optimum $20$ for this instance |

A greedy run compares 10 outgoing distances; a 2-opt scan evaluates five complete tours. These are different operations, not runtime measurements. Neither successful runs nor a small search effort prove a general global guarantee.

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 5 · Interactive comparison

The initial-tour slider selects the length-22 greedy tour (0) or the length-21 comparison tour (1). The neighborhood slider selects adjacent swaps (0) or full 2-opt (1). Hold the initial tour fixed while changing only the neighborhood. The scan slider shows each full neighbor scan; values beyond the final scan keep that final state. Teal bars improve the orange current-distance line. A complete scan with no teal bar establishes local optimality only for the selected neighborhood.

In [ ]:
import sys
import numpy as np
import matplotlib

if sys.platform != "emscripten":
    try:
        matplotlib.use("widget", force=True)
    except (RuntimeError, ValueError):
        from matplotlib.backends import backend_registry
        backend_registry._clear()
        matplotlib.use("widget", force=True)
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider

BLUE, TEAL, ORANGE, PURPLE, GRAY = "#2878b5", "#168578", "#e78b24", "#8854a5", "#707070"
plt.rcParams.update({"font.size": 11, "axes.titlesize": 11})

# Fixed travel distances; no map coordinates are assumed.
distances = np.array([[0,2,9,10,7], [2,0,8,5,6], [9,8,0,4,3],
                      [10,5,4,0,1], [7,6,3,1,0]])

def evaluate_tour(tour):
    tour = tuple(tour)
    feasible = (len(tour)==6 and set(tour[:-1])==set(range(5))
                and tour[-1]==tour[0])
    edges = [distances[a,b] for a,b in zip(tour[:-1], tour[1:])] if feasible else []
    return {"decision": tour, "feasible": feasible, "edges": edges,
            "objective": sum(edges) if feasible else None}

def neighbors(tour, full_two_opt):
    for i in range(1, 4):
        for j in range(i+1, 5):
            if (i==1 and j==4) or (not full_two_opt and j!=i+1):
                continue
            candidate = tour[:i] + tuple(reversed(tour[i:j+1])) + tour[j+1:]
            yield f"{i}:{j}", evaluate_tour(candidate)

def search(initial, full_two_opt):
    current, scans = evaluate_tour(initial), []
    while True:
        candidates = list(neighbors(current["decision"], full_two_opt))
        best = min(candidates, key=lambda item: item[1]["objective"])[1]
        improved = best["objective"] < current["objective"]
        scans.append({"current": current, "neighbors": candidates, "improved": improved})
        if not improved:
            return scans
        current = best

In [ ]:
figure, axis = plt.subplots(figsize=(5.6, 5.5))
figure.subplots_adjust(left=0.15, right=0.96, top=0.91, bottom=0.45)
status = figure.text(0.15, 0.30, "", va="top")
initial_slider = Slider(figure.add_axes([0.25, 0.18, 0.58, 0.028]), "initial tour",
                        0, 1, valinit=0, valstep=1, valfmt="%d", color=PURPLE)
rule_slider = Slider(figure.add_axes([0.25, 0.115, 0.58, 0.028]), "2-opt",
                     0, 1, valinit=1, valstep=1, valfmt="%d", color=PURPLE)
scan_slider = Slider(figure.add_axes([0.25, 0.05, 0.58, 0.028]), "scan",
                     1, 3, valinit=1, valstep=1, valfmt="%d", color=ORANGE)

def update(_):
    initial = [(1,0,4,3,2,1), (2,3,1,0,4,2)][int(initial_slider.val)]
    scans = search(initial, bool(rule_slider.val))
    index = min(int(scan_slider.val)-1, len(scans)-1)
    record = scans[index]
    current = record["current"]["objective"]
    labels = [label for label, result in record["neighbors"]]
    costs = [result["objective"] for label, result in record["neighbors"]]
    axis.clear()
    axis.bar(labels, costs, color=[TEAL if value<current else BLUE for value in costs])
    axis.axhline(current, color=ORANGE, label=f"Current: {current}")
    axis.set(xlabel="Reversed internal positions i:j", ylabel="Tour distance (units)",
             ylim=(0, 39), title="A full scan determines whether an improving move exists")
    axis.grid(axis="y", alpha=0.25)
    axis.legend(fontsize=10)
    tour_text = " → ".join(map(str, record["current"]["decision"]))
    outcome = f"Accept distance {min(costs)}" if record["improved"] else "Stop: no improving neighbor"
    status.set_text(f"Scan {index+1}/{len(scans)}; current tour: {tour_text}\n{outcome}")
    figure.canvas.draw_idle()

figure._sliders = [initial_slider, rule_slider, scan_slider]
for slider in figure._sliders:
    slider.on_changed(update)
update(None)
plt.show()